# Stance Active Labeling Inference

Batch workflow for the stance example. Shared code from `utils.py` and `plotting.py`.


In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

import numpy as np
import pandas as pd

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name in {'Stance', 'Alphafold', 'CheXpert', 'BRCA'} else CWD
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils import (
    binary_odds_ratio_truth,
    HUMAN_N_COL,
    EFFECTIVE_N_COL,
    run_odds_ratio_monte_carlo,
    summarize_monte_carlo,
)
from plotting import (
    make_monte_carlo_variance_table,
    plot_coverage,
    plot_effective_sample_size,
    plot_effective_sample_size_and_finite_population_coverage,
    plot_effective_sample_size_multiplier,
    plot_finite_population_coverage,
    plot_intervals,
    plot_monte_carlo_variance,
    plot_monte_carlo_variance_components,
    save_monte_carlo_variance_table,
)

import re
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split


In [2]:
EXAMPLE_DIR = REPO_ROOT / "Stance"
DATA_DIR = REPO_ROOT / "Data" / "Stance"
PLOTS_DIR = EXAMPLE_DIR / "plots"
RESULTS_DIR = EXAMPLE_DIR / "results"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 614
ALPHA = 0.1
TAU = 0.5
FRACS_HUMAN = np.linspace(0.2, 0.5, 10)
NUM_TRIALS = 500
TRAIN_PER_GROUP = 100


In [3]:
AFFIRMING_DEVICES = [
    "uncover", "realize", "know", "understand", "learn", "concede",
    "remember", "recall", "discover", "show", "reveal", "see",
    "forget", "find", "point out", "indicate", "acknowledge",
    "admit", "notice", "certify", "verify", "corroborate", "affirm",
    "confirm", "agree", "conclude", "proven", "settled", "conclusive",
    "definitive", "famed", "unequivocal", "skilful", "notable", "strong",
    "famous", "Nobel", "skillful", "Nobelist", "Nobel Laureate",
    "Nobel prize winner", "Nobel prize winning", "prize winning", "award",
    "winning", "distinguished", "well-grounded", "esteemed", "proficient",
    "key", "evidence", "noted", "top", "preeminent", "breakthrough",
    "significant", "intelligent", "of import", "celebrated", "novel", "recent",
    "major", "landmark", "important", "renowned", "peer-reviewed", "expert",
    "leading", "thousand", "1000", "hundred", "100", "unanimous", "diverse",
    "substantial", "many", "multiple", "dozen", "numerous",
]
LABEL_MAP = {"A": 1, "B": 0, "C": 0, "agrees": 1, "neutral": 0, "disagrees": 0}

raw_df = pd.read_csv(DATA_DIR / "stance_dataset.csv")
raw_df = raw_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
pattern = "|".join(re.escape(term) for term in AFFIRMING_DEVICES)
raw_df["contains_affirming_device"] = raw_df["sentence"].str.contains(
    pattern, case=False, regex=True
).fillna(False)

work = raw_df.dropna(subset=["confidence_in_prediction_gpt-4o", "label_gpt4o", "MACE_pred"]).copy()
work = work[
    work["label_gpt4o"].isin(LABEL_MAP)
    & work["MACE_pred"].isin(LABEL_MAP)
].copy()

Yhat = work["label_gpt4o"].map(LABEL_MAP).to_numpy(dtype=int)
Y = work["MACE_pred"].map(LABEL_MAP).to_numpy(dtype=int)
confidence = work["confidence_in_prediction_gpt-4o"].to_numpy(dtype=float).reshape(-1, 1)
device = work["contains_affirming_device"].to_numpy(dtype=bool)

Y0_full, Yhat0_full, confidence0_full = Y[~device], Yhat[~device], confidence[~device]
Y1_full, Yhat1_full, confidence1_full = Y[device], Yhat[device], confidence[device]

Y1_train, Y1_test, Yhat1_train, Yhat1_test, confidence1_train, confidence1_test = train_test_split(
    Y1_full,
    Yhat1_full,
    confidence1_full,
    train_size=TRAIN_PER_GROUP,
    random_state=SEED,
)
Y0_train, Y0_test, Yhat0_train, Yhat0_test, confidence0_train, confidence0_test = train_test_split(
    Y0_full,
    Yhat0_full,
    confidence0_full,
    train_size=TRAIN_PER_GROUP,
    random_state=SEED,
)

confidence_train = np.vstack([confidence1_train, confidence0_train])
Y_train = np.concatenate([Y1_train, Y0_train])
Yhat_train = np.concatenate([Yhat1_train, Yhat0_train])
residual_train = (Y_train - Yhat_train) ** 2
uncertainty_model = GradientBoostingRegressor(random_state=SEED, max_depth=2, n_estimators=100)
uncertainty_model.fit(confidence_train, residual_train)
uncertainty1 = np.sqrt(np.maximum(uncertainty_model.predict(confidence1_test), 1e-8))
uncertainty0 = np.sqrt(np.maximum(uncertainty_model.predict(confidence0_test), 1e-8))

true_odds_ratio, true_variance = binary_odds_ratio_truth(Y0_test, Y1_test)
mu0_pilot, mu1_pilot = float(np.mean(Y0_train)), float(np.mean(Y1_train))

pd.DataFrame(
    {
        "group": ["no affirming device", "affirming device"],
        "n_test": [len(Y0_test), len(Y1_test)],
        "outcome_mean": [Y0_test.mean(), Y1_test.mean()],
        "prediction_mean": [Yhat0_test.mean(), Yhat1_test.mean()],
        "uncertainty_mean": [uncertainty0.mean(), uncertainty1.mean()],
    }
)


,group,n_test,outcome_mean,prediction_mean,uncertainty_mean
0,no affirming device,1733,0.388344,0.517023,0.420703
1,affirming device,366,0.355191,0.513661,0.413449


In [4]:
df = run_odds_ratio_monte_carlo(
    y0=Y0_test,
    yhat0=Yhat0_test,
    y1=Y1_test,
    yhat1=Yhat1_test,
    fracs_human=FRACS_HUMAN,
    alpha=ALPHA,
    num_trials=NUM_TRIALS,
    true_odds_ratio=true_odds_ratio,
    true_variance=true_variance,
    mu0_pilot=mu0_pilot,
    mu1_pilot=mu1_pilot,
    tau=TAU,
    seed=SEED,
    uncertainty0=uncertainty0,
    uncertainty1=uncertainty1,
    spline_score0=uncertainty0,
    spline_score1=uncertainty1,
    active_budget="proportional",
    split_spline_budget_evenly=False,
    show_progress=True,
)

summary_df = summarize_monte_carlo(df)
mc_variance_table = make_monte_carlo_variance_table(df)

df.to_csv(RESULTS_DIR / "Stance_results.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "Stance_monte_carlo_summary.csv", index=False)
mc_variance_table.to_csv(RESULTS_DIR / "Stance_monte_carlo_variance_components.csv", index=False)

mc_variance_table.head(12)


human budget:   0%|          | 0/10 [00:00<?, ?it/s]

trials 0.200:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.233:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.267:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.300:   0%|          | 0/500 [00:00<?, ?it/s]

...

trials 0.400:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.433:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.467:   0%|          | 0/500 [00:00<?, ?it/s]

trials 0.500:   0%|          | 0/500 [00:00<?, ?it/s]

,$n_{\mathrm{human}}$,estimator,point_estimate_mean,point_estimate_variance,lb_mean,lb_variance,ub_mean,ub_variance,interval_width_mean,interval_width_variance,coverage,finite_population_coverage,finite_population_interval_width,finite_population_variance_inflation
0,419,active,0.891203,0.032669,0.613336,0.021379,1.297314,0.048111,0.683978,0.007050,0.936,0.894,0.576885,1.388239
1,419,active + tuning,0.857641,0.026151,0.606298,0.016353,1.214345,0.041967,0.608047,0.007436,0.946,0.876,0.497779,1.473425
2,419,classical,0.899973,0.045394,0.577792,0.020378,1.402534,0.102348,0.824742,0.032778,0.938,0.900,0.723764,1.300436
3,419,spline,0.872281,0.015387,0.648145,0.007809,1.174237,0.030924,0.526092,0.008363,0.970,0.866,0.390372,1.818861
4,419,spline + tuning,0.882002,0.015093,0.663377,0.008187,1.172864,0.028247,0.509487,0.006493,0.952,0.854,0.365845,1.934141
5,419,uniform,0.886532,0.033403,0.602097,0.021469,1.307917,0.050210,0.705820,0.007986,0.948,0.902,0.602989,1.352879
6,489,active,0.887448,0.024828,0.625076,0.017044,1.261627,0.034758,0.636551,0.004246,0.946,0.904,0.522270,1.468592
7,489,active + tuning,0.860410,0.018104,0.621130,0.011752,1.192654,0.028047,0.571524,0.004566,0.962,0.892,0.452638,1.575801
8,489,classical,0.900846,0.043267,0.598500,0.020717,1.356480,0.091124,0.757980,0.025970,0.938,0.892,0.656443,1.336707
9,489,spline,0.872800,0.010215,0.661433,0.005341,1.151918,0.019894,0.490485,0.005078,0.984,0.918,0.342134,2.060819


In [5]:
n_total = len(Y0_test) + len(Y1_test)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "Stance_effective_sample_size.pdf",
    n_total=n_total,
    error_bars="sd",
    show=False,
)
plot_effective_sample_size(
    df,
    path=PLOTS_DIR / "Stance_effective_sample_size_no_error_bars.pdf",
    n_total=n_total,
    error_bars="none",
    show=False,
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "Stance_effective_sample_size_multiplier.pdf",
    n_total=n_total,
    error_bars="sd",
    show=False,
)
plot_effective_sample_size_multiplier(
    df,
    path=PLOTS_DIR / "Stance_effective_sample_size_multiplier_no_error_bars.pdf",
    n_total=n_total,
    error_bars="none",
    show=False,
)
plot_effective_sample_size_and_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_effective_sample_size_and_corrected_coverage.pdf",
    n_total=n_total,
    show=False,
)
plot_effective_sample_size_and_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_effective_sample_size_and_corrected_coverage.png",
    n_total=n_total,
    show=False,
)
plot_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_coverage.pdf",
    n_total=n_total,
    show=False,
)
plot_finite_population_coverage(
    df,
    alpha=ALPHA,
    path=PLOTS_DIR / "Stance_coverage_finite_population_calibrated.pdf",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance(
    df,
    path=PLOTS_DIR / "Stance_monte_carlo_variance.pdf",
    n_total=n_total,
    show=False,
)
plot_monte_carlo_variance_components(
    df,
    path=PLOTS_DIR / "Stance_monte_carlo_variance_components.pdf",
    n_total=n_total,
    show=False,
)
save_monte_carlo_variance_table(
    df,
    path=PLOTS_DIR / "Stance_monte_carlo_variance_table.pdf",
    max_rows=18,
    show=False,
)
plot_intervals(
    df,
    true_value=true_odds_ratio,
    path=PLOTS_DIR / "Stance_intervals.pdf",
    estimand_label="odds ratio: affirming device vs none",
    show=False,
)


(<Figure size 700x840 with 1 Axes>,
 <Axes: xlabel='odds ratio: affirming device vs none'>)